In [9]:
%%bash
echo "fixing broken source line"
# Remove the faulty r2u sources configuration causing the warning
if [ -f /etc/apt/sources.list.d/r2u.sources ]; then
    rm -f /etc/apt/sources.list.d/r2u.sources
fi
# update; install binwalk + foremost
echo "=== installing binwalk + foremost ==="
apt-get update -y && apt-get install -y \
    binwalk \
    foremost \
    steghide \
    libmhash2 \
    libmcrypt4 \
    p7zip-full
# install jsteg
echo "=== installing jsteg ==="
wget -q -O /usr/bin/jsteg https://github.com
chmod +x /usr/bin/jsteg
wget -q -O /usr/bin/slink https://github.com
chmod +x /usr/bin/slink

# install stegseek
echo "=== installing stegseek ==="
wget -q https://github.com/RickdeJager/stegseek/releases/download/v0.6/stegseek_0.6-1.deb
apt-get install -y ./stegseek_0.6-1.deb &> /dev/null
rm -f ./stegseek_0.6-1.deb

#stegoveritas + dependencies
echo "installing stegoveritas"
pip install --upgrade pip &> /dev/null
pip install stegoveritas &> /dev/null
#note: stegoveritas_install_deps auto-downloads underlying tools like zsteg, exam, etc.
stegoveritas_install_deps &> /dev/null

echo "all tools installed successfully"

Process is interrupted.


In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Initialization; Setup

sample_dir = '/kaggle/working/sample_images'
report_dir = '/kaggle/working/forensics_reports'
carve_dir = '/kaggle/working/extracted_artifacts'
os.makedirs(report_dir, exist_ok=True)
os.makedirs(carve_dir, exist_ok=True)

# Path to standard dictionary for stegseek passphrase cracking
wordlist_path = '/usr/share/dict/words' 

images = sorted([f for f in os.listdir(sample_dir) if f.lower().endswith(('.jpg', '.jpeg'))])
print(f"Running the toolkit on {len(images)} JPEGs using updated JPEG-native toolkit...")

stats_records = []

# run the toolkit

for index, img_name in enumerate(images, 1):
    img_path = os.path.join(sample_dir, img_name)
    
    # Track category prefix
    category = "Unknown"
    for cat in ['Cover', 'JMiPOD', 'JUNIWARD', 'UERD']:
        if img_name.startswith(cat):
            category = cat
            break

    # Physical telemetry
    byte_size = os.path.getsize(img_path)
    
    # Forensic counters
    binwalk_hits = 0
    foremost_extracted_files = 0
    jsteg_anomaly = 0
    stegseek_cracked = 0

    print(f"[{index}/{len(images)}] Processing {img_name} ({category})...")

    # [Tool 1] BINWALK
    bw_res = subprocess.run(['binwalk', img_path], capture_output=True, text=True)
    if bw_res.stdout:
        lines = [l for l in bw_res.stdout.split('\n') if l.strip()]
        if len(lines) > 3:
            binwalk_hits = len(lines) - 3

    # [Tool 2] FOREMOST
    img_carve_out = os.path.join(carve_dir, img_name + "_carved")
    subprocess.run(['foremost', '-i', img_path, '-o', img_carve_out], capture_output=True)
    if os.path.exists(img_carve_out):
        carved_items = [f for f in os.listdir(img_carve_out) if f != 'audit.txt']
        foremost_extracted_files = len(carved_items)

    # [Tool 3] JSTEG (JPEG-Native LSB Analysis)
    # Attempts to extract hidden sequential arrays inside DCT coefficients
    jsteg_out_txt = os.path.join(carve_dir, img_name + "_jsteg.txt")
    js_res = subprocess.run(['jsteg', 'reveal', img_path, jsteg_out_txt], capture_output=True, text=True)
    # If file is successfully created and has contents, flag it
    if os.path.exists(jsteg_out_txt) and os.path.getsize(jsteg_out_txt) > 0:
        jsteg_anomaly = 1

    # [Tool 4] STEGSEEK
    if os.path.exists(wordlist_path):
        ss_res = subprocess.run(['stegseek', '--wordlist', wordlist_path, img_path], capture_output=True, text=True)
        if "Found passphrase" in ss_res.stderr or "Cracked" in ss_res.stdout:
            stegseek_cracked = 1

    # [Tool 5] STEGOVERITAS
    sv_out = os.path.join(carve_dir, img_name + "_veritas")
    subprocess.run(['stegoveritas', img_path, '-out', sv_out], capture_output=True)

    # Log metrics
    stats_records.append({
        "filename": img_name,
        "class": category,
        "group": "Cover" if category == "Cover" else "Stego",
        "file_size_bytes": byte_size,
        "binwalk_hits": binwalk_hits,
        "carved_files_count": foremost_extracted_files,
        "jsteg_anomaly": jsteg_anomaly,
        "stegseek_success": stegseek_cracked
    })

# Save structured CSV
df = pd.DataFrame(stats_records)
df.to_csv(f"{report_dir}/forensic_statistical_matrix.csv", index=False)
print(f"Forensic processing done. Data matrix exported.")

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')